In [ ]:
%load_ext dotenv
%dotenv

In [2]:
from utils import neo4j_driver, num_tokens_from_string, chunk_text, chat, embed
import ch07_tools

import json
import requests

from tqdm import tqdm
from typing import List, Dict

In [3]:
url = "https://www.gutenberg.org/cache/epub/1727/pg1727.txt"
response = requests.get(url)

In [4]:
# save response content to a file
with open("pg1727.txt", "w", encoding="utf-8") as f:
    f.write(response.text)

In [5]:
def chunk_into_books(text: str) -> List[str]:
    return (
        text.split("PREFACE TO FIRST EDITION")[2]
        .split("FOOTNOTES")[0]
        .strip()
        .split("\nBOOK")[1:]
    )

books = chunk_into_books(response.text)

In [6]:
token_count = [num_tokens_from_string(el) for el in books]
print(
    f"""There are {len(token_count)} books with token sizes:
- avg {sum(token_count) / len(token_count)}
- min {min(token_count)}
- max {max(token_count)}
"""
)

There are 24 books with token sizes:
- avg 6466.625
- min 4421
- max 10701



In [7]:
chunked_books = [chunk_text(book, 1000, 40) for book in books]

In [8]:
ENTITY_TYPES = [
    "PERSON",
    "ORGANIZATION",
    "LOCATION",
    "GOD",
    "EVENT",
    "CREATURE",
    "WEAPON_OR_TOOL",
]
def extract_entities(text: str) -> List[Dict]:
    # Construct prompt
    messages = [
        {"role": "user", "content": ch07_tools.create_extraction_prompt(ENTITY_TYPES, text)},
    ]
    # Make the LLM call
    output = chat(messages)
    # Construct JSON from output
    return ch07_tools.parse_extraction_output(output)

In [11]:
number_of_books = 1
for book_i, book in enumerate(
    tqdm(chunked_books[:number_of_books], desc="Processing Books")
):
    for chunk_i, chunk in enumerate(tqdm(book, desc=f"Book {book_i}", leave=False)):
        nodes, relationships = extract_entities(chunk)
        neo4j_driver.execute_query(
            ch07_tools.import_nodes_query,
            {
                "data": nodes,
                "book_id": book_i,
                "text": chunk,
                "chunk_id": chunk_i,
            }
        )
        neo4j_driver.execute_query(
            ch07_tools.import_relationships_query,
            {
                "data": relationships
            }
        )

Processing Books:   0%|          | 0/1 [00:00<?, ?it/s]Transaction failed and will be retried in 0.941708041758147s (Couldn't connect to localhost:7687 (resolved to ('[::1]:7687', '127.0.0.1:7687')):
Connection to [::1]:7687 closed with incomplete handshake response
Connection to 127.0.0.1:7687 closed with incomplete handshake response)
Transaction failed and will be retried in 1.7504547195361093s (Couldn't connect to localhost:7687 (resolved to ('[::1]:7687', '127.0.0.1:7687')):
Connection to [::1]:7687 closed with incomplete handshake response
Connection to 127.0.0.1:7687 closed with incomplete handshake response)
Transaction failed and will be retried in 4.222182010562079s (Couldn't connect to localhost:7687 (resolved to ('[::1]:7687', '127.0.0.1:7687')):
Connection to [::1]:7687 closed with incomplete handshake response
Connection to 127.0.0.1:7687 closed with incomplete handshake response)
Processing Books: 100%|██████████| 1/1 [01:26<00:00, 86.52s/it]


In [12]:
data, _, _ = neo4j_driver.execute_query(
    """MATCH (:`__Entity__`)
    RETURN 'entity' AS type, count(*) AS count
    UNION
    MATCH ()-[:RELATIONSHIP]->()
    RETURN 'relationship' AS type, count(*) AS count
    """
)
print([el.data() for el in data])

[{'type': 'entity', 'count': 124}, {'type': 'relationship', 'count': 298}]


In [13]:
data, _, _ = neo4j_driver.execute_query(
    """MATCH (n:PERSON)
WHERE n.name = "ORESTES"
RETURN n.description AS description"""
)
print([el.data()['description'] for el in data])

[["Orestes is Agamemnon's son who killed Aegisthus", 'Orestes is a person who would take revenge when grown up', "Orestes is a figure in Greek mythology, mentioned as an example of avenging a father's murder", 'Orestes is Agamemnon’s son who killed Aegisthus', 'Orestes is a person who would take revenge when grown up', "Orestes is a person mentioned in the text as someone praised for killing his father's murderer"]]


In [14]:
data, _, _ = neo4j_driver.execute_query(
    """MATCH (n:__Entity__)-[:RELATIONSHIP]-(m:__Entity__)
WITH n,m, count(*) AS countOfRels
ORDER BY countOfRels DESC LIMIT 1
MATCH (n)-[r:RELATIONSHIP]-(m)
RETURN n.name AS source, m.name AS target, countOfRels, collect(r.description) AS descriptions
"""
)
print([el.data() for el in data])

[{'source': 'TELEMACHUS', 'target': 'MINERVA', 'countOfRels': 11, 'descriptions': ['Telemachus receives counsel from Minerva regarding his voyage', 'Minerva disguised herself as a stranger to guide and encourage Telemachus', 'Telemachus speaks privately with Minerva', 'Telemachus speaks privately with Minerva', 'Minerva disguised herself as a stranger to guide and encourage Telemachus', 'Telemachus receives counsel from Minerva regarding his voyage', 'Minerva follows Telemachus to his house to guide and test him', 'Minerva intends to embolden Telemachus to confront the suitors', 'Minerva intends to embolden Telemachus to confront the suitors', 'Minerva follows Telemachus to his house to guide and test him', "Minerva shows interest in Telemachus' well-being"]}]


In [ ]:
import time

candidates_to_summarize, _, _ = neo4j_driver.execute_query(
    """MATCH (e:__Entity__) WHERE size(e.description) > 1 
    RETURN e.name AS entity_name, e.description AS description_list"""
)
summaries = []
for candidate in tqdm(candidates_to_summarize, desc="Summarizing entities"):
    messages = [
        {
            "role": "user",
            "content": ch07_tools.get_summarize_prompt(
                candidate["entity_name"], candidate["description_list"]
            ),
        },
    ]
    # pause 1 second between calls to avoid rate limits
    time.sleep(1)
    summary = chat(messages)
    summaries.append({"entity": candidate["entity_name"], "summary": summary})

ch07_tools.import_entity_summary(neo4j_driver, summaries)

Summarizing entities: 100%|██████████| 28/28 [00:58<00:00,  2.10s/it]


In [15]:
summary, _, _ = neo4j_driver.execute_query(
    """MATCH (n:PERSON)
WHERE n.name = "ORESTES"
RETURN n.summary AS summary""")
print(summary[0]['summary'])

Orestes is a figure in Greek mythology, the son of Agamemnon, who avenged his father's murder by killing Aegisthus. As he grew up, he became a person known for taking revenge when he reached adulthood.


In [ ]:
rels_to_summarize, _, _ = neo4j_driver.execute_query(
    """MATCH (s:__Entity__)-[r:RELATIONSHIP]-(t:__Entity__)
    WHERE id(s) < id(t)
    WITH s.name AS source, t.name AS target, 
           collect(r.description) AS description_list,
           count(*) AS count
    WHERE count > 1
    RETURN source, target, description_list"""
)
rel_summaries = []
for candidate in tqdm(rels_to_summarize, desc="Summarizing relationships"):
    entity_name = f"{candidate['source']} relationship to {candidate['target']}"
    messages = [
        {
            "role": "user",
            "content": ch07_tools.get_summarize_prompt(
                entity_name, candidate["description_list"]
            ),
        },
    ]
    # pause in between calls to avoid rate limits
    time.sleep(0.5)
    summary = chat(messages)
    rel_summaries.append({"source": candidate["source"], "target": candidate["target"], "summary": summary})

ch07_tools.import_rels_summary(neo4j_driver, summaries)

Summarizing relationships: 100%|██████████| 17/17 [00:25<00:00,  1.48s/it]


In [16]:
data, _, _ = neo4j_driver.execute_query(
    """MATCH (n:__Entity__)-[r:SUMMARIZED_RELATIONSHIP]-(m:__Entity__)
WHERE n.name = 'TELEMACHUS' AND m.name = 'MINERVA'
RETURN r.summary AS description
"""
)
print(data[0]["description"])

Minerva intends to embolden Telemachus to confront the suitors


In [17]:
from importlib import reload
reload(ch07_tools)

<module 'ch07_tools' from 'c:\\Users\\Shira\\NTUST\\Agentic AI\\kg-rag\\notebooks\\ch07_tools.py'>

In [18]:
community_distribution = ch07_tools.calculate_communities(neo4j_driver)
print(f"There are {community_distribution['communityCount']} communities with distribution: {community_distribution['communityDistribution']}")

There are 9 communities with distribution: {'p1': 4, 'p5': 4, 'max': 23, 'p90': 23, 'p50': 10, 'p95': 23, 'p10': 4, 'p75': 14, 'p99': 23, 'p25': 7, 'min': 4, 'mean': 11.666666666666666, 'p999': 23}


In [20]:
community_info, _, _ = neo4j_driver.execute_query(ch07_tools.community_info_query)

communities = []
for community in tqdm(community_info, desc="Summarizing communities"):
    messages = [
        {
            "role": "user",
            "content": ch07_tools.get_summarize_community_prompt(
                community["nodes"], community["rels"]
            ),
        },
    ]
    summary = chat(messages)
    communities.append(
        {
            "community": json.loads(ch07_tools.extract_json(summary)),
            "communityId": community["communityId"],
            "nodes": [el["id"] for el in community["nodes"]],
        }
    )

neo4j_driver.execute_query(ch07_tools.import_community_query, data=communities)

[#F5BB]  _: <CONNECTION> error: Failed to read from defunct connection IPv4Address(('localhost', 7687)) (ResolvedIPv6Address(('::1', 7687, 0, 0))): ConnectionAbortedError(10053, 'Eine bestehende Verbindung wurde softwaregesteuert\r\ndurch den Hostcomputer abgebrochen', None, 10053, None)
Transaction failed and will be retried in 1.0566498787177465s (Failed to read from defunct connection IPv4Address(('localhost', 7687)) (ResolvedIPv6Address(('::1', 7687, 0, 0))))
Transaction failed and will be retried in 2.3585964770582866s (Couldn't connect to localhost:7687 (resolved to ('[::1]:7687', '127.0.0.1:7687')):
Failed to read four byte Bolt handshake response from server ResolvedIPv6Address(('::1', 7687, 0, 0)) (deadline Deadline(timeout=60.0))
Connection to 127.0.0.1:7687 closed with incomplete handshake response)
Transaction failed and will be retried in 4.274158804366641s (Couldn't connect to localhost:7687 (resolved to ('[::1]:7687', '127.0.0.1:7687')):
Failed to read four byte Bolt han

EagerResult(records=[], summary=<neo4j._work.summary.ResultSummary object at 0x00000269BA816F90>, keys=[])

In [21]:
data, _, _ = neo4j_driver.execute_query(
    """MATCH (c:__Community__)
WITH c, count {(c)<-[:IN_COMMUNITY]-()} AS size
ORDER BY size DESC LIMIT 1
RETURN c.title AS title, c.summary AS summary
"""
)
print(data[0]["title"])
print(data[0]["summary"])

Odysseus' Household and the Suitors of Ithaca
The community centers around the household of Odysseus in Ithaca, where his son Telemachus and wife Penelope are struggling to maintain control against a group of suitors who are exploiting the household's resources. The suitors, led by Antinous, are vying for Penelope's hand in marriage and the throne, while Telemachus seeks to assert his authority and news of his father's whereabouts.


In [22]:
def global_retriever(query: str, rating_threshold: float = 5) -> str:
    community_data, _, _ = neo4j_driver.execute_query(
        """
    MATCH (c:__Community__)
    WHERE c.rating >= $rating
    RETURN c.summary AS summary
    """,
        rating=rating_threshold,
    )
    print(f"Got {len(community_data)} community summaries")
    intermediate_results = []
    for community in tqdm(community_data, desc="Processing communities"):
        intermediate_messages = [
            {
                "role": "system",
                "content": ch07_tools.get_map_system_prompt(community["summary"]),
            },
            {
                "role": "user",
                "content": query,
            },
        ]
        intermediate_response = chat(intermediate_messages)
        intermediate_results.append(intermediate_response)

    final_messages = [
        {
            "role": "system",
            "content": ch07_tools.get_reduce_system_prompt(intermediate_results),
        },
        {"role": "user", "content": query},
    ]
    summary = chat(final_messages)
    return summary

In [23]:
print(global_retriever("What is this story about?"))

Got 7 community summaries


Processing communities: 100%|██████████| 7/7 [00:15<00:00,  2.25s/it]


The story is a tapestry of Greek mythology and legend, woven around the themes of divine intervention, mortal struggle, and the enduring quest for homecoming. At its core, it follows the trials of Ulysses (Odysseus), the legendary Greek hero whose journey home from the Trojan War is fraught with challenges, both divine and mortal [Data: Reports (1, 2, 3, 4, 5)]. His voyage is marked by detainment by the nymph Calypso, encounters with the sun god Hyperion’s sacred cattle, and the wrath of Poseidon (Neptune), who prolongs his suffering at sea [Data: Reports (2, 4, 5, 1)]. Meanwhile, back in Ithaca, his household is besieged by suitors vying for his wife Penelope’s hand and his throne, while his son Telemachus strives to assert his authority and uncover news of his father’s fate [Data: Reports (1, 3, 4)].

The narrative is deeply intertwined with the machinations of the gods, particularly Minerva (Athena), who serves as a patron and protector to Ulysses and Telemachus. She appears in disg

In [ ]:
import time

entities, _, _ = neo4j_driver.execute_query(
    """
MATCH (e:__Entity__)
RETURN e.summary AS summary, e.name AS name
"""
    )

data = []
for el in entities:
    if el["summary"] is not None:
        embedding = embed(el["summary"])[0]
        data.append({"name": el["name"], "embedding": embedding})
        time.sleep(0.5)  # Pause 0.5 seconds between API calls to avoid rate limits

In [ ]:
neo4j_driver.execute_query(
    """
UNWIND $data AS row
MATCH (e:__Entity__ {name: row.name})
SET e.embedding = row.embedding
""",
{"data": data}
)

neo4j_driver.execute_query(
    """
CREATE VECTOR INDEX entities IF NOT EXISTS
FOR (n:__Entity__)
ON (n.embedding)
"""
)


EagerResult(records=[], summary=<neo4j._work.summary.ResultSummary object at 0x00000269B9B15820>, keys=[])

In [34]:
local_search_query = """
CALL db.index.vector.queryNodes('entities', $k, $embedding)
YIELD node, score
WITH collect(node) as nodes
WITH collect {
    UNWIND nodes as n
    MATCH (n)<-[:HAS_ENTITY]->(c:__Chunk__)
    WITH c, count(distinct n) as freq
    RETURN c.text AS chunkText
    ORDER BY freq DESC
    LIMIT $topChunks
} AS text_mapping,
collect {
    UNWIND nodes as n
    MATCH (n)-[:IN_COMMUNITY]->(c:__Community__)
    WITH c, c.rank as rank, c.weight AS weight
    RETURN c.summary 
    ORDER BY rank, weight DESC
    LIMIT $topCommunities
} AS report_mapping,
collect {
    UNWIND nodes as n
    MATCH (n)-[r:SUMMARIZED_RELATIONSHIP]-(m) 
    WHERE m IN nodes
    RETURN r.summary AS descriptionText
    ORDER BY r.rank, r.weight DESC 
    LIMIT $topInsideRels
} as insideRels,
collect {
    UNWIND nodes as n
    RETURN n.summary AS descriptionText
} as entities
RETURN {Chunks: text_mapping, Reports: report_mapping, 
       Relationships: insideRels, 
       Entities: entities} AS text
"""

In [37]:
k_entities = 5

topChunks = 3
topCommunities = 3
topInsideRels = 3


def local_search(query: str) -> str:
    context, _, _ = neo4j_driver.execute_query(
        local_search_query,
        embedding=embed(query)[0],
        topChunks=topChunks,
        topCommunities=topCommunities,
        topInsideRels=topInsideRels,
        k=k_entities,
    )
    context_str = str(context[0]["text"])
    local_messages = [
        {
            "role": "system",
            "content": ch07_tools.get_local_system_prompt(context_str),
        },
        {
            "role": "user",
            "content": query,
        },
    ]
    final_answer = chat(local_messages)
    return final_answer


In [38]:
print(local_search("Who is Ulysses?"))

Ulysses, also known as Odysseus in Greek mythology, is a legendary figure celebrated for his cunning, resourcefulness, and pivotal role in the Trojan War. As the king of Ithaca, he is best known for his long and arduous journey home after the war, a voyage immortalized in Homer’s *Odyssey*. His absence from Ithaca and the trials he faced—including encounters with gods, monsters, and divine figures—became central themes in his mythological legacy. Ulysses is depicted as an ingenious hero navigating a lonely, sea-girt island, symbolizing perseverance, strategy, and the enduring bonds of home and family [Data: Entities (1)].

Ulysses' journey home was fraught with challenges, including his infamous encounter with Polyphemus, the Cyclops son of Neptune, whom he blinded. This act of defiance incurred Neptune’s wrath, complicating Ulysses’ voyage further [Data: Relationships (1); Entities (3)]. His story is intertwined with divine and mortal figures, such as Calypso, who detained him on her 

In [39]:
local_search("Who is POLYBUS?")

"Polybus is a figure in Greek mythology, specifically mentioned as the father of Eurymachus, one of the suitors in Ithaca who is consuming Odysseus' estate during his absence [Data: Entities (1, 2); Relationships (1, 3)]. The data suggests that Polybus is part of the political and social tensions in Ithaca, where suitors like Eurymachus are exploiting the resources of Odysseus' household [Data: Reports (1, 3)]. While the provided data does not elaborate further on Polybus' role or background, his connection to Eurymachus situates him within the broader narrative of the *Odyssey*, where the absence of Odysseus leads to such challenges in his homeland."